# Análise exploratória — Cursos de graduação presenciais (Brasil e 12 maiores capitais, 2000–2022)
**Autores:** João Victor Teixeira · Renan Braga Gomes — Bacharelado em Sistemas de Informação, UniLaSalle (6º período, 2026.2)
**Disciplina:** Linguagem de Programação · **Professor:** Alexandre Louzada
**Fonte:** INEP via Data.Rio, Tabela 913.

> Execute `python -m src.etl` antes (gera `data/processed/`). As funções reutilizáveis estão em `src/etl.py`.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, numpy as np
import plotly.express as px
from pathlib import Path
from src import etl

PROC = Path("../data/processed")
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Aquisição e parsing do arquivo bruto
O `.xls` tem uma aba por ano, cabeçalho institucional e rodapé. Inspecionamos duas abas antes de parsear.

In [ ]:
raw = Path("../data/raw/913.xls")
xl = pd.ExcelFile(raw, engine="xlrd")
print(xl.sheet_names[:3], "...", xl.sheet_names[-2:])
xl.parse("2022", header=None).head(14)   # note '35 765' como texto em 2022

In [ ]:
longo, prov, descartadas = etl.carregar_planilha(raw)
print("Abas descartadas:", descartadas)
longo.head(14)

## 2. Qualidade dos dados
Checagens: nulos, duplicatas, soma das capitais = agregado, percentual oficial vs. recalculado.

In [ ]:
import json
rel = etl.validar(longo)
print(json.dumps(rel, ensure_ascii=False, indent=2))

## 3. Métricas derivadas

In [ ]:
m = etl.construir_metricas(longo)
res = etl.resumo_por_localidade(m)
conc = etl.indicadores_concentracao(m)
res

## 4. RQ1 — Evolução temporal e inflexões

In [ ]:
d = m[m.localidade.isin(["Brasil", "12 Capitais"])]
px.line(d, x="ano", y="total_cursos", color="localidade", markers=True,
        title="Cursos de graduação presenciais", labels={"total_cursos": "Cursos"})

In [ ]:
px.bar(m[m.localidade == "Brasil"].dropna(subset=["var_pct_anual"]), x="ano", y="var_pct_anual",
       title="Variação anual do total nacional (%)")

## 5. RQ2 — Participação das capitais e concentração

In [ ]:
px.line(m[m.localidade == "12 Capitais"], x="ano", y="pct_brasil_calc", markers=True,
        title="Participação das 12 capitais no total nacional (%)")

In [ ]:
px.line(conc, x="ano", y=["gini", "cv"], title="Dispersão entre capitais: Gini e coeficiente de variação")

## 6. RQ3 — Quem cresceu mais? Convergência

In [ ]:
cap = res[res.tipo == "capital"]
px.scatter(cap, x="cursos_2000", y="cagr_pct", text="localidade", log_x=True,
           trendline="ols", title="Nível inicial (2000) vs. CAGR 2000–2022")

In [ ]:
from scipy import stats
print(stats.spearmanr(cap.cursos_2000, cap.cagr_pct))
print(stats.pearsonr(np.log(cap.cursos_2000), cap.cagr_pct))

## 7. Outliers e heatmap de variações

In [ ]:
etl.sinalizar_outliers(m).head(10)

In [ ]:
h = m[m.tipo == "capital"].pivot(index="localidade", columns="ano", values="var_pct_anual")
px.imshow(h, color_continuous_scale="RdBu", zmin=-30, zmax=30, aspect="auto",
          title="Variação anual (%) por capital")

## 8. Conclusões preliminares
- Expansão forte até 2009, desaceleração depois, queda em 2020–2021 e recuperação parcial em 2022.
- Participação das capitais estável (≈ 26–29%), com degrau em 2009.
- Desconcentração: capitais menores cresceram mais rápido que as maiores (ρ de Spearman ≈ −0,69).
- Rio de Janeiro teve o menor CAGR entre as capitais.